# Notebook 10 . Cocokkan Nama yang Tidak Persis Sama

Bantu mencari pasangan nama yang mirip, misalnya nama pelanggan yang ejaannya sedikit berbeda. Hasilnya adalah daftar kandidat untuk ditinjau manusia; kecocokan tidak diterapkan otomatis karena skor kemiripan bisa salah.

**Pustaka tambahan:** jalankan `pip install rapidfuzz` pada lingkungan Python notebook jika belum tersedia. Fuzzy matching dapat memakan waktu pada file besar; mulai dari sampel kecil.

In [ ]:
from pathlib import Path
import pandas as pd
try:
    from rapidfuzz import process, fuzz
except ImportError as exc:
    raise ImportError("Pustaka rapidfuzz belum tersedia. Pasang dengan: pip install rapidfuzz") from exc

# PENGATURAN YANG BISA DIUBAH
Input_CSV_Kiri = Path("Input/sumber_a.csv")
Input_CSV_Kanan = Path("Input/sumber_b.csv")
Kolom_Nama_Kiri = "nama"
Kolom_Nama_Kanan = "nama"
Kolom_ID_Kiri = "id"       # Bisa None jika tidak ada kolom ID
Kolom_ID_Kanan = "id"      # Bisa None jika tidak ada kolom ID
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma
Skor_Minimum = 85        # Skor 0-100; naikkan jika ingin hasil lebih ketat
Selisih_Skor_Minimum = 5           # Selisih minimal terhadap kandidat kedua; turunkan untuk melihat kandidat ambigu
Batas_Baris = None     # Contoh 500 untuk uji sampel; None untuk semua baris
Output_CSV = Path("Output/hasil_kandidat_pencocokan.csv")

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Cari kandidat terbaik untuk ditinjau

In [ ]:
def read_input(path):
    path = path if path.is_absolute() else BASE_DIR / path
    if not path.exists(): raise FileNotFoundError(f"File tidak ditemukan: {path}")
    return pd.read_csv(path, dtype="string", encoding="utf-8-sig", sep=Separator_CSV)
left, right = read_input(Input_CSV_Kiri), read_input(Input_CSV_Kanan)
for frame, col in [(left, Kolom_Nama_Kiri), (right, Kolom_Nama_Kanan)]:
    if col not in frame: raise KeyError(f"Kolom nama tidak ditemukan: {col}")
if Kolom_ID_Kiri and Kolom_ID_Kiri not in left: raise KeyError(f"Kolom ID kiri tidak ditemukan: {Kolom_ID_Kiri}")
if Kolom_ID_Kanan and Kolom_ID_Kanan not in right: raise KeyError(f"Kolom ID kanan tidak ditemukan: {Kolom_ID_Kanan}")
if Batas_Baris is not None: left = left.head(Batas_Baris)
right = right.copy()
right["_match_text"] = right[Kolom_Nama_Kanan].fillna("").astype(str).str.casefold().str.strip()
choices = right["_match_text"].tolist()
results = []
for _, row in left.iterrows():
    original = "" if pd.isna(row[Kolom_Nama_Kiri]) else str(row[Kolom_Nama_Kiri]).strip()
    query = original.casefold()
    hits = process.extract(query, choices, scorer=fuzz.WRatio, limit=2) if query else []
    best = hits[0] if hits else None
    second_score = hits[1][1] if len(hits) > 1 else -1
    idx = best[2] if best else None
    score = best[1] if best else 0
    candidate = right.iloc[idx] if idx is not None else None
    accepted_candidate = bool(best and score >= Skor_Minimum and score - second_score >= Selisih_Skor_Minimum)
    item = {"nama_kiri": original, "nama_kandidat": candidate[Kolom_Nama_Kanan] if candidate is not None else pd.NA,
            "skor": score, "skor_kandidat_kedua": second_score,
            "saran": "periksa kandidat" if accepted_candidate else "perlu tinjauan manual"}
    if Kolom_ID_Kiri: item["id_kiri"] = row[Kolom_ID_Kiri]
    if Kolom_ID_Kanan: item["id_kandidat"] = candidate[Kolom_ID_Kanan] if candidate is not None else pd.NA
    results.append(item)
result = pd.DataFrame(results)
display(result.head(30))
print(f"Kandidat dibuat: {len(result):,}. Semua hasil perlu diperiksa sebelum dipakai untuk mengubah data.")


In [ ]:
output = Output_CSV if Output_CSV.is_absolute() else BASE_DIR / Output_CSV
output.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output, index=False, encoding="utf-8-sig", sep=Separator_CSV)
print(f"Tersimpan: {output.resolve()}")